Classificação de Texto com Deep Learning

## 1.Importação dos dados
Importação do conjunto de dados do IMDB, separados em treino e teste


In [ ]:
import tensorflow_datasets as tfds
dataset, info = tfds.load('imdb_reviews', with_info=True,
                          as_supervised=True)
train_dataset, test_dataset = dataset['train'], dataset['test']

train_dataset.element_spec

In [ ]:
# Imprime algumas amostras do dataset
for example, label in train_dataset.take(5):
  print('text: ', example.numpy())
  print('label: ', label.numpy())

## 2. Preparação dos dados

In [ ]:
# Extração de textos e labels de the train_dataset
all_train_texts = []
all_train_labels = []

for text_item, label_item in train_dataset:
    all_train_texts.append(text_item.numpy().decode('utf-8'))
    all_train_labels.append(label_item.numpy())

print(f"Número de amostras: {len(all_train_texts)}")
print(f"Textos: {all_train_texts[:5]}")
print(f"Labels: {all_train_labels[:5]}")

### 2.2. Camada de vetorização

Esta camada possui opções básicas para gerenciar texto em um modelo Keras.  Ela transforma um lote de strings (um exemplo = uma string) em uma lista  de índices de tokens (um exemplo = tensor 1D de índices de tokens inteiros) ou em uma representação densa (um exemplo = tensor 1D de valores float  representando dados sobre os tokens do exemplo).  Esta camada destina-se a lidar com entradas em linguagem natural.


In [ ]:
from tensorflow.keras.layers import TextVectorization
import numpy as np

# Camada de vetorização: TextVectorization.

imdb_vectorize_layer = TextVectorization(
    max_tokens=10000, # Tamanho do vocabulário
    output_mode='int',
    output_sequence_length=256 # Tamanho máximo do texto
)


### 2.3. Processamento
O processamento de cada exemplo contém as seguintes etapas:

1. Padronizar cada exemplo (geralmente convertendo para minúsculas e removendo a pontuação)
2. Dividir cada exemplo em substrings (geralmente palavras)
3. Recombinar as substrings em tokens (geralmente n-gramas)
4. Indexar os tokens (associar um valor inteiro único a cada token)
5. Transformar cada exemplo usando esse índice, seja em um vetor de inteiros ou em um vetor denso de números de ponto flutuante.

In [ ]:
# O vocabulário da camada deve ser fornecido na construção ou aprendido
# por meio da função `adapt()
imdb_vectorize_layer.adapt(np.array(all_train_texts))

# Transform the IMDB training texts into padded integer sequences
X = imdb_vectorize_layer(np.array(all_train_texts)).numpy()

# Assign the extracted IMDB labels to y
y = np.array(all_train_labels)

print(f"Shape of X (processed IMDB texts): {X.shape}")
print(f"Shape of y (IMDB labels): {y.shape}")

In [ ]:
# Extract all texts and labels from the test_dataset
all_test_texts = []
all_test_labels = []

for text_item, label_item in test_dataset:
    all_test_texts.append(text_item.numpy().decode('utf-8'))
    all_test_labels.append(label_item.numpy())

print(f"Number of test examples: {len(all_test_texts)}")

# Transform the IMDB test texts into padded integer sequences using the same vectorization layer
X_test = imdb_vectorize_layer(np.array(all_test_texts)).numpy()

# Assign the extracted IMDB test labels to y_test
y_test = np.array(all_test_labels)

print(f"Shape of X_test (processed IMDB test texts): {X_test.shape}")
print(f"Shape of y_test (IMDB test labels): {y_test.shape}")

## 3. Definindo a rede neural
A rede neural pode ser vista como uma sequencia de camadas de transformação, a saída de uma camada é dada como entrada para a próxima camada

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Flatten, Dense

model = Sequential([
    ## reduz a dimensão do vetor de entrada da cama de vetorização
    Embedding(input_dim=10000, output_dim=16, input_length=256),
    ## transformação de um tensor multidimensional  em um tensor unidimensional,
    # geralmente para preparar os dados para camadas totalmente conectadas
    # (densas).
    Flatten(),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

In [ ]:
model.summary()

## 4. Treinamento do modelo


In [ ]:
import tensorflow as tf
import numpy as np

# O método compile configura um modelo Keras para treinamento, especificando
# seu otimizador, função de perda e métricas de desempenho.
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# Converte X e y para tensores no TensorFlow
X_tensor = tf.constant(X, dtype=tf.int32)
y_tensor = tf.constant(np.array(y), dtype=tf.float32)


model.fit(X_tensor, y_tensor, epochs=10, verbose=1)

In [ ]:
# Evaluate the model on the test data
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)

print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

In [ ]:
model.summary()

In [ ]:
new_texts = [
    "This movie was fantastic! I loved every minute of it.",
    "Absolute garbage. A complete waste of time and money.",
    "It was an okay movie, nothing special but not bad either."
]

vectorized_new_texts = imdb_vectorize_layer(np.array(new_texts))


predictions = model.predict(vectorized_new_texts)

print("\n--- Predictions ---")
for i, prediction in enumerate(predictions):
    sentiment = "Positive" if prediction > 0.5 else "Negative"
    print(f"Text: \"{new_texts[i]}\"")
    print(f"Prediction: {prediction[0]:.4f} (Sentiment: {sentiment})")
